In [1]:
import json
from transformers import AutoTokenizer

# This ONLY loads the tiny tokenizer files (~5MB)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

print("✅ Tokenizer loaded successfully without the model!")

# Let's test a single sentence to see it in action
test_text = "The committee remains highly attentive to inflation risks."
tokens = tokenizer.encode(test_text)

print(f"Text: {test_text}")
print(f"Token IDs: {tokens}")
print(f"Count: {len(tokens)} tokens")

c:\Users\Javier\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tokenizer loaded successfully without the model!
Text: The committee remains highly attentive to inflation risks.
Token IDs: [1, 1183, 14182, 8288, 7184, 1766, 1076, 1263, 1066, 21381, 15447, 29491]
Count: 12 tokens


In [4]:
import json
import numpy as np
from transformers import AutoTokenizer

# 1. Setup
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
DATA_PATH = "data/all_unlabelled_sentences/master_unlabelled_pool.json"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 2. Load the data
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

qa_lengths = []
print(f"📊 Analyzing {len(data)} items for token distribution...")

# 3. Processing
for item in data:
    ans = item.get("sentence", "")
    ques = item.get("context_question", "")
    
    # We simulate the exact format the model will see during training
    # Including the reporter's question and the Chair's answer
    text = f"Question: {ques} Answer: {ans}"
    
    # encode() turns text into a list of numbers; len() counts them
    count = len(tokenizer.encode(text))
    qa_lengths.append(count)

# 4. Final Results
p99 = int(np.percentile(qa_lengths, 99))
max_val = max(qa_lengths)

print("\n" + "="*30)
print(f"99th Percentile: {p99} tokens")
print(f"Absolute Max:    {max_val} tokens")
print("="*30)

if p99 <= 256:
    print("\n✅ Recommendation: Keep max_seq_length: 256 in your YAML.")
elif p99 <= 512:
    print("\n⚠️ Recommendation: Increase max_seq_length to 512 in your YAML.")
else:
    print("\n🚨 Recommendation: You have some very long texts! Consider 1024.")

📊 Analyzing 117061 items for token distribution...

99th Percentile: 216 tokens
Absolute Max:    425 tokens

✅ Recommendation: Keep max_seq_length: 256 in your YAML.


In [3]:
# Calculate the exact count of sentences that exceed 256 tokens
count_over_256 = sum(1 for length in qa_lengths if length > 256)
percentage_over = (count_over_256 / len(qa_lengths)) * 100

print(f"\nTotal sentences > 256 tokens: {count_over_256}")
print(f"Percentage of dataset truncated: {percentage_over:.2f}%")


Total sentences > 256 tokens: 457
Percentage of dataset truncated: 0.39%
